# Merging Text Files for Corpus Building

This notebook was originally created for use on the Maastricht DSRI and merges several `.txt` files into one to help you build a corpus for text analysis. The files are joined in the order you choose, one after another.

**Note:** This version runs entirely in your browser via Live Code (Pyodide). There is no access to a folder on your computer. You upload your files below and download the merged file at the end. Nothing is saved automatically, and everything is gone when you close the tab.

---

## Step 1: Import Required Libraries

We only need tools for talking to your browser (file upload and download). No `ipywidgets` and no local folder are involved.

In [ ]:
from IPython.display import display, HTML
from js import document, Blob, URL, Object
from pyodide.ffi import create_proxy, to_js

## Step 2: Upload Your Files

Run the cell below. A **Choose files** button appears. Select two or more `.txt` files from your computer. Hold **Ctrl** (Windows) or **Cmd** (Mac) to select several files in the file dialog.

Choosing again **replaces** the previous selection, so select all the files you want to merge in one go.

Files are read as UTF-8 text, which is what most text editors save by default.

In [ ]:
uploaded_files = {}   # filename -> full text of the file, in the order the browser provides them

display(HTML('<div id="merge-upload-box"><input type="file" id="merge-upload" accept=".txt" multiple> '
             '<span id="merge-upload-status">No files selected yet.</span></div>'))

file_input = document.getElementById("merge-upload")
status = document.getElementById("merge-upload-status")

if file_input is None:
    # Fallback: add the controls to the page if the output was not yet in the DOM
    box = document.createElement("div")
    file_input = document.createElement("input")
    file_input.type = "file"
    file_input.accept = ".txt"
    file_input.multiple = True
    status = document.createElement("span")
    status.textContent = " No files selected yet."
    box.appendChild(file_input)
    box.appendChild(status)
    document.body.appendChild(box)

async def _read_files(event):
    uploaded_files.clear()
    n = file_input.files.length
    if n == 0:
        status.textContent = "No files selected yet."
        return
    for i in range(n):
        f = file_input.files.item(i)
        uploaded_files[f.name] = await f.text()
    status.textContent = f"Loaded {len(uploaded_files)} file(s). Continue with Step 3."

file_input.addEventListener("change", create_proxy(_read_files))

## Step 3: Choose Order and Output Name

The merged file contains the files one after another. Your browser often hands over the files in alphabetical order, regardless of the order in which you clicked them, so you can set the order explicitly here.

- `output_name`: the name of the merged file (change it as you like)
- `sort_files`: `"upload"` keeps the order the browser provided, `"name"` sorts alphabetically
- `custom_order`: optional. Write the exact file names in the order you want, for example `["file2.txt", "file1.txt", "file3.txt"]`. If you fill this in, it overrides `sort_files`.

In [ ]:
output_name = "my_merged_files.txt"
sort_files = "upload"      # "upload" or "name"
custom_order = []          # e.g. ["file2.txt", "file1.txt", "file3.txt"]

if not uploaded_files:
    print("No files loaded yet. Go back to Step 2 and upload your files first.")
    merge_order = []
else:
    if custom_order:
        unknown = [n for n in custom_order if n not in uploaded_files]
        if unknown:
            print("These names in custom_order were not found among the uploaded files:", unknown)
            print("Uploaded files:", list(uploaded_files.keys()))
            merge_order = []
        else:
            merge_order = list(custom_order)
    elif sort_files == "name":
        merge_order = sorted(uploaded_files.keys())
    else:
        merge_order = list(uploaded_files.keys())

    if merge_order:
        print("Files will be merged in this order:")
        for i, name in enumerate(merge_order, start=1):
            print(f"  {i}. {name}")

## Step 4: Merge the Files

The text of all files is joined into one. If a file does not end with a line break, one is added, so the last line of one file never gets glued to the first line of the next.

In [ ]:
merged_text = ""

if not merge_order:
    print("Nothing to merge yet. Complete Steps 2 and 3 first.")
else:
    parts = []
    total_lines = 0
    for name in merge_order:
        text = uploaded_files[name]
        if text and not text.endswith("\n"):
            text += "\n"
        n_lines = len(text.splitlines())
        total_lines += n_lines
        print(f"{name}: {n_lines} lines")
        parts.append(text)
    merged_text = "".join(parts)
    print(f"Merged {len(merge_order)} files into {total_lines} lines in total.")
    print("Done!")

## Step 5: Download the Merged File

Because this notebook runs inside your browser, there is no folder to write to. Running the cell below creates a download link. Click it to save the merged file to your computer.

In [ ]:
if not merged_text:
    print("Nothing to download yet. Complete Steps 2 to 4 first.")
else:
    blob = Blob.new([merged_text], to_js({"type": "text/plain;charset=utf-8"}, dict_converter=Object.fromEntries))
    url = URL.createObjectURL(blob)

    display(HTML('<div id="merge-download-box"></div>'))
    box = document.getElementById("merge-download-box")
    if box is None:
        box = document.body

    link = document.createElement("a")
    link.href = url
    link.download = output_name
    link.textContent = f"Download {output_name}"
    box.appendChild(link)
    print("Click the link above to save your merged file.")